## [Tutorial](https://github.com/biohub/esm/tree/main/cookbook/tutorials): How to run minibinder + scFv design on Google Colab

In this notebook, we run end-to-end de novo binder design directly on **Google Colab's GPU runtime** using the protocol described in the ESMC and ESMFold2 paper: ["Language Modeling Materializes a World Model of Protein Biology"](https://www.biorxiv.org/content/10.64898/2026.06.03.729735).

Biohub used this approach to design minibinders and scFvs against five therapeutically relevant targets — PDGFRB, EGFR, PD-L1, CD45, and CTLA4 — spanning receptor tyrosine kinases, immune checkpoints, and cell-surface phosphatases. Binders exhibit nanomolar affinity, target specificity, and functional activity in laboratory assays.

---

### Hardware Requirements
- **Runtime:** Make sure you are using a GPU runtime (**Runtime > Change runtime type > GPU**).
- **Recommended GPU:** An **A100 GPU** or **L4 GPU** with High-RAM is recommended. For single T4 GPUs (15 GB VRAM), keep `batch_size=1` and `use_scaling_critics=False`.

**Workflow:**
1. **Setup**: install dependencies, verify Colab GPU, download `binder_design.py`
2. **Try one job**: pick a target and binder type, run a single design end-to-end locally
3. **Run a sweep**: evaluate multiple candidate seeds sequentially on Colab
4. **Pick the designs to order**: filter by isoelectric point, score by iPTM, and export PDB structures directly to disk/Google Drive.

### 1. Setup

In [ ]:
# Environment setup for Colab
! pip install -q esm py3dmol pyarrow biopython abnumber
# Lightweight modal stub so binder_design.py imports without errors
! pip install -q modal
# Install hmmer system dependency (required if designing antibodies via abnumber)
! apt-get update -qq && apt-get install -y -qq hmmer

In [ ]:
# Check GPU availability and specifications
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Active GPU: {device_name} ({vram_gb:.2f} GB VRAM)")
    if vram_gb < 20:
        print("Note: Running on <= 16GB VRAM (e.g. T4). Keep batch_size=1 and use_scaling_critics=False.")
    else:
        print("Sufficient VRAM detected for ESMFold2 + ESMC-6B.")
else:
    print("WARNING: No GPU detected! Go to Runtime -> Change runtime type and select a GPU (A100/L4/T4).")

### Download the design script

Download `binder_design.py` into your Colab session. This module defines the optimization loop, loss functions, prompt factories, and ESMFold2 model handling.

Run the cell below to fetch `binder_design.py` directly from the Biohub ESM repository into your current Colab working directory.

In [ ]:
# Download binder_design.py into the working directory
! wget -q https://raw.githubusercontent.com/Biohub/esm/main/cookbook/tutorials/binder_design.py
! ls -lh binder_design.py

In [ ]:
# Verify that binder_design is present and can be imported locally
import binder_design
print("Successfully loaded binder_design module!")

### Imports

In [ ]:
import gc
from itertools import product
from pathlib import Path

import pandas as pd
import py3Dmol
import torch
from Bio.SeqUtils.ProtParam import ProteinAnalysis
from tqdm.auto import tqdm

# Import ESMFold2Design directly from local binder_design module (no Modal needed)
from binder_design import ESMFold2Design

### Model initialization on Colab GPU

Instead of spinning up remote Modal cloud instances, we instantiate `ESMFold2Design()` directly inside your Colab runtime.

- `use_scaling_critics=False` (**strongly recommended on Colab**): skips the additional 15-checkpoint ensemble from the paper, dramatically saving GPU VRAM and download time while keeping the core hero inversion and critic models alongside ESMC-6B.
- `use_scaling_critics=True`: loads the full 15-checkpoint ensemble (use only if running on an A100 80GB instance with High-RAM).

In [ ]:
# Initialize ESMFold2Design directly on Colab GPU
app = ESMFold2Design()

# Load models (use_scaling_critics=False is recommended for standard Colab GPU memory)
app.load(use_scaling_critics=False)
print("ESMFold2 and ESMC models successfully loaded into Colab GPU memory!")

## 2. Try one design job

Run a single design job end-to-end directly on your Colab GPU as a sanity check.

Pick **one** of the two options below and run only that cell:
- **Option 1**: uses built-in target presets (`ctla4`, `egfr`, `pdgfrb`, `pd-l1`, `cd45`) and binder scaffolds (`minibinder`, `trastuzumab_framework_vhvl`).
- **Option 2**: takes your own custom target and binder sequence scaffold (where `#` means "design this mutable position").

The design runs synchronously on your Colab GPU, printing optimization loss progress every 5 steps.

In [ ]:
# ---- Option 1: Use presets (Direct Colab Execution) ----
torch.cuda.empty_cache()
gc.collect()

best_sequences, trajectory, critic_results = app.design(
    target_name="ctla4",
    binder_name="minibinder",
    batch_size=1,
    seed=0,
)
print("Design complete!")
print("Best sequence:", best_sequences[0])

In [ ]:
# ---- Option 2: Provide your own sequences (Direct Colab Execution) ----
torch.cuda.empty_cache()
gc.collect()

# Example: PD-L1 sequence crop
pdl1_sequence = "AFTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKNIIQFVHGEEDLKVQHSSYRQRARLLKDQLSLGNAALQITDVKLQDAGVYRCMISYGGADYKRITVKVNA"
# Trastuzumab-style antibody scaffold
trastuzumab_framework_vhvl = "EVQLVESGGGLVQPGGSLRLSCAAS#######YIHWVRQAPGKGLEWVARI#####TRYADSVKGRFTISADTSKNTAYLQMNSLRAEDTAVYYCSR###########WGQGTLVTVSSGGGSGGGSGGGSGGGSDIQMTQSPSSLSASVGDRVTITC###########WYQQKPGKAPKLLIY#######GVPSRFSGSRSGTDFTLTISSLQPEDFATYYC#########FGQGTKVEIK"

best_sequences, trajectory, critic_results = app.design(
    target_name="pd-l1-custom",
    target_sequence=pdl1_sequence,
    binder_name="trastuzumab-custom",
    binder_sequence=trastuzumab_framework_vhvl,
    is_antibody=True,
    batch_size=1,
    seed=0,
)
print("Design complete!")
print("Best sequence:", best_sequences[0])

In [ ]:
# ---- Trajectory Metrics ----
# In Colab, optimization progress is logged in real-time above.
# Here we can inspect the recorded trajectory and final step loss breakdown:
final_step = max(trajectory.keys())
print(f"Total optimization steps recorded: {len(trajectory)}")
print(f"Final step ({final_step}) losses:")
for metric_name, val in trajectory[final_step].items():
    numeric_val = val.item() if hasattr(val, "item") else val
    print(f"  {metric_name}: {numeric_val:.4f}")

In [ ]:
# ---- Load and Inspect Results ----
df = pd.DataFrame(critic_results)
print("Best designed sequence:\n", best_sequences[0])
display(df.drop(columns=["logits", "complex"], errors="ignore"))

In [ ]:
# ---- Visualize Predicted Complex in 3D ----
if "complex" in df.columns and any(df["complex"].notna()):
    cutoff_match = df[df.critic_name.str.contains("Cutoff2025", na=False)]
    protein_complex = cutoff_match.iloc[0].complex if len(cutoff_match) > 0 else df.iloc[0].complex

    view = py3Dmol.view(width=600, height=600)
    view.addModel(protein_complex.to_pdb_string(), "pdb")
    view.setStyle({"chain": "A"}, {"cartoon": {"color": "green"}})
    view.setStyle(
        {"chain": "B"},
        {
            "cartoon": {
                "colorscheme": {"prop": "b", "gradient": "rwb", "min": 60, "max": 100}
            }
        },
    )
    view.addStyle(  # B-factor coloring for binder
        {"and": [{"chain": "B"}, {"not": {"atom": ["N", "C", "O"]}}]},
        {
            "stick": {
                "colorscheme": {"prop": "b", "gradient": "rwb", "min": 60, "max": 100},
                "radius": 0.2,
            }
        },
    )
    view.addStyle(  # Target colored green
        {"and": [{"chain": "A"}, {"not": {"atom": ["N", "C", "O"]}}]},
        {"stick": {"color": "green", "radius": 0.2}},
    )
    view.center()
    view.zoomTo()
    view.show()
else:
    print("No complex structure found in results.")

## 3. Run a sweep for real designs

To generate a diverse pool of candidate binders, sweep across multiple random seeds directly on your Colab GPU.

In Colab, jobs run sequentially on your local GPU. We recommend starting with a smaller seed list (e.g., 4–8 seeds with `batch_size=1`) to confirm runtime speed before scaling up.

**Before running the sweep, review the configuration below.**

In [ ]:
# ---- Sweep Configuration ----
save_dir = Path("sweep")
save_dir.mkdir(exist_ok=True)

targets = [("pd-l1", None)]
binders = [("minibinder", None)]

line_sweeps = dict(
    target=targets,
    binder=binders,
    use_scaling_critics=[False],
    seed=list(range(4)),   # Start with 4 seeds on Colab; increase as desired
    batch_size=[1],        # batch_size=1 recommended for single-GPU Colab stability
)
df = pd.DataFrame(product(*line_sweeps.values()), columns=line_sweeps.keys())
df["target_name"], df["target_sequence"] = zip(*df["target"], strict=True)
df["binder_name"], df["binder_sequence"] = zip(*df["binder"], strict=True)
df = df.drop(columns=["target", "binder"])
display(df.head())
print(f"Total design jobs to run on Colab: {len(df)}")

In [ ]:
# ---- Execute Sweep on Colab GPU ----
sweep_results = []

for row in tqdm(df.itertuples(), total=len(df), desc="Running design sweep"):
    torch.cuda.empty_cache()
    gc.collect()

    seqs, traj, critic_res = app.design(
        target_name=row.target_name,
        target_sequence=row.target_sequence,
        binder_name=row.binder_name,
        binder_sequence=row.binder_sequence,
        seed=row.seed,
        batch_size=row.batch_size,
    )
    for r in critic_res:
        item = dict(r)
        item["target_name"] = row.target_name
        item["binder_name"] = row.binder_name
        item["seed"] = row.seed
        sweep_results.append(item)

df_all = pd.DataFrame(sweep_results)
df_all.drop(columns=["logits", "complex"], errors="ignore").to_parquet(save_dir / "manifest.parquet", index=False)
print(f"\nSuccessfully finished {len(df)} jobs! Manifest saved to {save_dir / 'manifest.parquet'}")

### Persisting Results to Google Drive (Recommended)

Google Colab runtimes automatically disconnect after inactivity, clearing `/content/`. To permanently keep your designs, sequences, and structures, mount Google Drive before running long sweeps:

```python
from google.colab import drive
drive.mount('/content/drive')
save_dir = Path('/content/drive/MyDrive/esm_binder_sweep')
save_dir.mkdir(parents=True, exist_ok=True)
```

You can then reload previously generated results anytime without recomputing them.

In [ ]:
# ---- Inspect Sweep Results ----
print(f"Total critic evaluations across sweep: {len(df_all)}")
display(df_all.drop(columns=["logits", "complex"], errors="ignore").head())

Since all jobs executed locally on your Colab GPU, all completed design results are already consolidated in memory and stored in `df_all`.

In [ ]:
# ---- Prepare Successful Candidates ----
df_success = df_all.copy()
print(f"Total entries: {len(df_success)}")
print(f"Unique designed sequences: {df_success['designed_sequence'].nunique()}")

## 4. Pick the designs to order

This is your final shortlist. The cell below:

1. Combines results from all successful jobs into one dataframe.
2. Filters minibinders to isoelectric point under 6 (helps with solubility and expression). Antibodies pass through unfiltered.
3. Scores each unique designed sequence by averaging `iptm` (interface predicted TM-score, higher is better) and an `iptm_proxy` term across its trajectories.
4. Returns the top 84 designs per (target, binder type), saved to `selection.parquet` inside your `save_dir`.
5. Writes one PDB per selected design to `selected_structures/` inside your `save_dir`.

84 is a plate-friendly number for ordering and screening. The cutoff and the isoelectric-point filter are currently hardcoded inside the cell, so to change them, edit the values directly in the function.


In [ ]:
# ---- Select Top Candidates ----

df_result = df_success.copy()

# Filter minibinder designs with isoelectric point >= 6 (solubility & expression)
df_result["binder_sequence"] = df_result.designed_sequence.str.split(r"\|").str[1]
df_result["isoelectric_point"] = [
    ProteinAnalysis(seq).isoelectric_point()
    for seq in tqdm(df_result.binder_sequence.values, desc="Calculating pI")
]
df_filter = df_result[df_result.is_antibody | df_result.isoelectric_point.lt(6)]

SCALING_CHECKPOINT_SUBSTRING = "ESMFold2-Experimental-Fast-base"

def select(group_df: pd.DataFrame) -> pd.DataFrame:
    d = group_df.copy()
    is_scaling = d.critic_name.str.contains(
        SCALING_CHECKPOINT_SUBSTRING, regex=False, na=False
    )
    if "distogram_iptm_proxy" in d.columns:
        iptm_proxy = d.distogram_iptm_proxy.where(
            ~d.is_antibody, d.get("cdr_distogram_iptm_proxy", d.distogram_iptm_proxy)
        )
    else:
        iptm_proxy = d.iptm

    d["iptm_score"] = d.iptm.where(~is_scaling, 0.0)
    d["iptm_proxy_score"] = iptm_proxy.where(is_scaling, 0.0)
    scores = d.groupby("designed_sequence", as_index=False).agg(
        iptm_score=("iptm_score", "mean"), iptm_proxy_score=("iptm_proxy_score", "mean")
    )
    scores["selection_score"] = 0.5 * scores.iptm_score.fillna(0) + 0.5 * scores.iptm_proxy_score.fillna(0)
    return scores.nlargest(min(len(scores), 84), "selection_score")

df_select = df_filter.groupby(["target_name", "binder_name"]).apply(
    select, include_groups=False
)
df_select.to_parquet(save_dir / "selection.parquet", index=False)
display(df_select)

In [ ]:
# ---- Write Selected Structures to PDB ----
pdb_dir = save_dir / "selected_structures"
pdb_dir.mkdir(exist_ok=True)

# Find rows containing predicted complex structures
valid_complexes = df_result[df_result["complex"].notna()]
if len(valid_complexes) > 0:
    # Prefer Cutoff2025 checkpoint if available, otherwise take first available complex
    cutoff_match = valid_complexes[valid_complexes.critic_name.str.contains("Cutoff2025", na=False)]
    complex_src = cutoff_match if len(cutoff_match) > 0 else valid_complexes

    complexes = (
        complex_src.drop_duplicates("designed_sequence")
        .set_index("designed_sequence")["complex"]
    )

    selected = df_select.reset_index().sort_values(
        ["target_name", "binder_name", "selection_score"], ascending=[True, True, False]
    )
    written_count = 0
    for rank, row in enumerate(selected.itertuples(), start=1):
        if row.designed_sequence in complexes:
            pdb_path = (
                pdb_dir
                / f"{rank:04d}_{row.target_name}_{row.binder_name}_score{row.selection_score:.3f}.pdb"
            )
            pdb_path.write_text(complexes[row.designed_sequence].to_pdb_string())
            written_count += 1
    print(f"Successfully wrote {written_count} PDB structures to {pdb_dir.resolve()}")
else:
    print("Note: No complex structures found to write.")

In [ ]:
df_result.drop(columns=["complex", "logits"], errors="ignore").head(10)

In [ ]:
df_select

## Appendix

### Google Colab GPU Optimization Guide

- **GPU Selection**  
  In Colab, go to **Runtime > Change runtime type** and select a GPU. An **A100 (40GB/80GB)** or **L4 (24GB)** provides ample memory to comfortably run the full `ESMC-6B` + `ESMFold2` optimization loop.

- **VRAM Optimization on T4 (15GB)**  
  If running on a standard T4 GPU:
  - Keep `use_scaling_critics=False` when calling `app.load()`.
  - Always keep `batch_size=1`.
  - Run `torch.cuda.empty_cache()` and `gc.collect()` between runs.

- **Mounting Google Drive**  
  Colab instances are ephemeral and reset if disconnected. Mount Google Drive (`from google.colab import drive; drive.mount('/content/drive')`) and save your output shortlists and PDB structures directly to `/content/drive/MyDrive/...`.

- **Monitoring Execution**  
  Optimization logs print directly to the cell output every 5 steps showing the gradient descent progression, intra/inter contacts, and overall loss.